# Sentiment Analysis on Hugging Face Financial Phrasebank

In this homework you will perform sentiment analysis on the *Financial Phrasebank* data set from Hugging Face.  In particular, you will use the Multinomial Naive Bayes model as well as another model of your choosing.  This is an individual assignment and your deliverable should be one or more Jupyter notebooks.

The Financial Phrase bank data set consists of 4,846 financial headlines that have been labeled with the following sentiments: `0` - negative; `1` - neutral; `2` - positive.  More information about the data set can be found here: https://huggingface.co/datasets/financial_phrasebank.

1. (5 pts) Devise a non-machine learning baseline against which you can assess whether your models have any predictive power.

2. (15 pts) Use the `CountVectorizer` preprocessor along with `MultinomialNB` to perform sentiment analysis on the data set.  Address the following:
   
    a. How will you measure out-of-sample performance of the models?
   
    b. Compared to the baseline, does the model seem to have predictive power?
   
    c. Experiment with the following parameters of the model: `ngram_range`, `stop_words`, `binary`.
   
    d. Do any of the above parameters affect the model's performance?
  
3. (15 pts) Use the `TfidVectorizer` preprocessor along with `MultinomialNB` to perform sentiment analysis on the data set.  Address the following:
   
    a. How will you measure out-of-sample performance of the models?
   
    b. Compared to the baseline, does the model seem to have predictive power?
   
    c. Experiment with the following parameters of the model: `ngram_range`, `stop_words`, `binary`.
   
    d. Do any of the above parameters affect the model's performance?
  
4. (15 pts) Use the **spaCy** word embedding model, along with a supervised model of your choosing, to perform sentiment analysis.  Address the following:

    a. How will you measure out-of-sample performance of the models?
   
    b. Compared to the baseline, does the model seem to have predictive power?

    c. How does the word-embedding based model compare to the `MultinomialNB` model. 

In [26]:
import numpy as np
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [27]:
## LOADing DATA FROM CSV
df = pd.read_csv("financial_phrasebank.csv")
df

,sentence,label
0,"According to Gran , the company has no plans t...",1
1,Technopolis plans to develop in stages an area...,1
2,The international electronic industry company ...,0
3,With the new production plant the company woul...,2
4,According to the company 's updated strategy f...,2
...,...,...
4841,LONDON MarketWatch -- Share prices ended lower...,0
4842,Rinkuskiai 's beer sales fell by 6.5 per cent ...,1
4843,Operating profit fell to EUR 35.4 mn from EUR ...,0
4844,Net sales of the Paper segment decreased to EU...,0


In [28]:
## Defining FEATURES AND TARGET, TRAIN/TEST SPLIT

X = df["sentence"].values        # raw text (headlines/sentences)
y = df["label"].values           # numeric labels: 0/1/2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

len(X_train), len(X_test)

(3392, 1454)

## Baseline: always predict the most frequent label in the training set.

In [29]:
from collections import Counter

# Find majority class in the training set
class_counts = Counter(y_train)
majority_label, majority_count = class_counts.most_common(1)[0]
print("Majority label in training set:", majority_label) 
print("count:", majority_count)

# Predict majority label for every test example
y_pred_baseline = np.full_like(y_test, fill_value=majority_label)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline (always predict {majority_label}) accuracy on test set: {baseline_accuracy:.4f}")

Majority label in training set: 1
count: 2015
Baseline (always predict 1) accuracy on test set: 0.5942


## CountVectorizer + MultinomialNB

CountVectorizer to get bag-of-words features, then MultinomialNB on those, and test on the held-out test set for the out of sample testing.

In [30]:
def run_nb_model(ngram_range=(1, 1), stop_words=None, binary=False):
    """
    Train MultinomialNB on CountVectorizer features with the given params,
    return train & test accuracy.
    """
    vect = CountVectorizer(
        ngram_range=ngram_range,
        stop_words=stop_words,
        binary=binary
    )
    X_train_vec = vect.fit_transform(X_train)
    X_test_vec  = vect.transform(X_test)

    clf = MultinomialNB()
    clf.fit(X_train_vec, y_train)

    y_train_pred = clf.predict(X_train_vec)
    y_test_pred  = clf.predict(X_test_vec)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)

    return train_acc, test_acc

In [31]:
train_acc_nb, test_acc_nb = run_nb_model(ngram_range=(1, 1), stop_words=None, binary=False)

print(f"MultinomialNB + CountVectorizer (unigrams): "f"train={train_acc_nb:.4f}, test={test_acc_nb:.4f}")
print(f"Baseline accuracy: {baseline_accuracy:.4f}")

MultinomialNB + CountVectorizer (unigrams): train=0.8871, test=0.7180
Baseline accuracy: 0.5942


### We can see the predictive power of this model is much higher than Baseline model

## Hyperparameter exploration: ngram_range, stop_words, binary

In [32]:
settings = [
    {"ngram_range": (1, 1), "stop_words": None,       "binary": False},
    {"ngram_range": (1, 1), "stop_words": "english",  "binary": False},
    {"ngram_range": (1, 2), "stop_words": None,       "binary": False},
    {"ngram_range": (1, 2), "stop_words": "english",  "binary": False},
    {"ngram_range": (1, 1), "stop_words": None,       "binary": True},
    {"ngram_range": (1, 2), "stop_words": None,       "binary": True},
    {"ngram_range": (1, 2), "stop_words": "english",  "binary": True},
]

In [33]:
results_nb = []

for cfg in settings:
    train_acc, test_acc = run_nb_model(**cfg)
    results_nb.append((cfg, train_acc, test_acc))
    print(f"NB, ngram_range={cfg['ngram_range']}, "f"stop_words={cfg['stop_words']}, binary={cfg['binary']} "f"-> train={train_acc:.4f}, test={test_acc:.4f}")

NB, ngram_range=(1, 1), stop_words=None, binary=False -> train=0.8871, test=0.7180
NB, ngram_range=(1, 1), stop_words=english, binary=False -> train=0.8942, test=0.7118
NB, ngram_range=(1, 2), stop_words=None, binary=False -> train=0.9702, test=0.7256
NB, ngram_range=(1, 2), stop_words=english, binary=False -> train=0.9708, test=0.7105
NB, ngram_range=(1, 1), stop_words=None, binary=True -> train=0.8939, test=0.7290
NB, ngram_range=(1, 2), stop_words=None, binary=True -> train=0.9744, test=0.7311
NB, ngram_range=(1, 2), stop_words=english, binary=True -> train=0.9741, test=0.7146


In [34]:
best_nb_test = max(r[2] for r in results_nb)
print("\nBest NB test accuracy across settings:", best_nb_test)


Best NB test accuracy across settings: 0.7310866574965612


### The best model overall: ngram_range = (1,2), stop_words = None, binary = True - Test accuracy = 0.7311. 

### Removing stopwords generally hurts NB: Models with stop_words="english" consistently perform worse than models that keep stopwords.

### Binary features (presence/absence) slightly improve performance. Binary bag-of-words improves generalization by reducing the effect of large term counts affecting NB likelihoods.

## TfidVectorizer + MultinomialNB

Here again I am using the Hold out set for test accuracy just like in part 2

In [35]:
def run_tfidf_nb_model(ngram_range=(1, 1), stop_words=None, min_df=1):
    """
    MultinomialNB on TF-IDF features.
    Mirrors run_nb_model but swaps CountVectorizer -> TfidfVectorizer.
    """
    vect = TfidfVectorizer(
        ngram_range=ngram_range,
        stop_words=stop_words,
        min_df=min_df
    )
    X_train_vec = vect.fit_transform(X_train)
    X_test_vec  = vect.transform(X_test)

    clf = MultinomialNB()
    clf.fit(X_train_vec, y_train)

    y_train_pred = clf.predict(X_train_vec)
    y_test_pred  = clf.predict(X_test_vec)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)

    return train_acc, test_acc

In [36]:
tfidf_settings = [
    {"ngram_range": (1, 1), "stop_words": None,      "min_df": 1},
    {"ngram_range": (1, 1), "stop_words": "english", "min_df": 1},
    {"ngram_range": (1, 2), "stop_words": None,      "min_df": 1},
    {"ngram_range": (1, 2), "stop_words": "english", "min_df": 1},
    {"ngram_range": (1, 2), "stop_words": None,      "min_df": 2},
    {"ngram_range": (1, 2), "stop_words": "english", "min_df": 2},
]

In [37]:
results_tfidf_nb = []

for cfg in tfidf_settings:
    train_acc, test_acc = run_tfidf_nb_model(**cfg)
    results_tfidf_nb.append((cfg, train_acc, test_acc))
    print(
        f"TFIDF+NB, ngram_range={cfg['ngram_range']}, "
        f"stop_words={cfg['stop_words']}, min_df={cfg['min_df']} "
        f"-> train={train_acc:.4f}, test={test_acc:.4f}"
    )

TFIDF+NB, ngram_range=(1, 1), stop_words=None, min_df=1 -> train=0.7300, test=0.6768
TFIDF+NB, ngram_range=(1, 1), stop_words=english, min_df=1 -> train=0.7583, test=0.6809
TFIDF+NB, ngram_range=(1, 2), stop_words=None, min_df=1 -> train=0.7624, test=0.6657
TFIDF+NB, ngram_range=(1, 2), stop_words=english, min_df=1 -> train=0.8004, test=0.6692
TFIDF+NB, ngram_range=(1, 2), stop_words=None, min_df=2 -> train=0.7762, test=0.6953
TFIDF+NB, ngram_range=(1, 2), stop_words=english, min_df=2 -> train=0.7810, test=0.6953


In [38]:
best_tfidf_nb_test = max(r[2] for r in results_tfidf_nb)
print("\nBest TF-IDF+NB test accuracy across settings:", best_tfidf_nb_test)
print("Baseline accuracy:", baseline_accuracy)
print("Best CountVectorizer+NB test accuracy (Part2):", best_nb_test)


Best TF-IDF+NB test accuracy across settings: 0.6953232462173315
Baseline accuracy: 0.594222833562586
Best CountVectorizer+NB test accuracy (Part2): 0.7310866574965612


### As we can see the Tfidf performance is much better than baseline but still lesser than the Countvertorizer (as visible in the result above)

### The best model overall: ngram_range = (1,2), min_df = 2, Test accuracy = 0.6953.

### Models with stop_words="english" perform almost the same as models that keep stopwords. The impact is small and does not consistently help or hurt generalization.

### Without rare-word filtering, (1,2) models overfit train accuracy rises but test accuracy drops. With min_df=2, bigrams become useful and deliver the best overall performance.

## SpaCy + LR (Supervised model)

In [39]:
nlp = spacy.load("en_core_web_lg")
nlp

In [40]:
def sentence_vector(text: str) -> np.ndarray:
    """
    Compute mean token vector for one sentence using spaCy.
    Handles empty values safely.
    """
    if not isinstance(text, str) or not text.strip():
        return np.zeros(nlp.vocab.vectors_length, dtype="float32")
    doc = nlp(text)
    if len(doc) == 0:
        return np.zeros(nlp.vocab.vectors_length, dtype="float32")
    return np.array([token.vector for token in doc]).mean(axis=0)

In [41]:
# Computing embeddings for train and test separately

X_train_emb = np.vstack([sentence_vector(s) for s in X_train])
X_test_emb  = np.vstack([sentence_vector(s) for s in X_test])

X_train_emb.shape, X_test_emb.shape

((3392, 300), (1454, 300))

In [42]:
clf_spacy = LogisticRegression(
    max_iter=500,
    solver="lbfgs",
    n_jobs=1
)

clf_spacy.fit(X_train_emb, y_train)

y_train_pred_spacy = clf_spacy.predict(X_train_emb)
y_test_pred_spacy  = clf_spacy.predict(X_test_emb)

train_acc_spacy = accuracy_score(y_train, y_train_pred_spacy)
test_acc_spacy  = accuracy_score(y_test,  y_test_pred_spacy)

In [43]:
print(f"spaCy embeddings + LogisticRegression: "f"train={train_acc_spacy:.4f}, test={test_acc_spacy:.4f}")
print("Baseline accuracy:", baseline_accuracy)
print("Best CountVectorizer+NB test accuracy (Q2):", best_nb_test)
print("Best TF-IDF+NB test accuracy (Q3):", best_tfidf_nb_test)

spaCy embeddings + LogisticRegression: train=0.7824, test=0.7345
Baseline accuracy: 0.594222833562586
Best CountVectorizer+NB test accuracy (Q2): 0.7310866574965612
Best TF-IDF+NB test accuracy (Q3): 0.6953232462173315


### As we see the SpaCy word embedding based model performs better than baseline and TfidfVectorizer + MultinomialNB as well as slightly better than CountVectorizer + MultinomialNB